In [1]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:

import numpy as np
import pandas as pd
import datetime
import random





In [ ]:
import pyodbc as db
#bad data digit query
con1 = db.connect('DRIVER={SQL Server}; SERVER=USSLCSQL31;DATABASE=KUCC_Piezo_PROD')  # connect to server
cur1 = con1.cursor()

qry1 = '''SELECT [ID_PIEZO]
      ,[TM_STAMP]
      ,[DIGITS]
      ,[AMP]
      ,[REPORTABLE]
  FROM [KUCC_Piezo_PROD].[dbo].[DIGIT_READINGS]
WHERE [TM_STAMP] between '9/1/2021'and '12/31/2022' AND [VAL_STATUS]='Validated' '''
# line orinigally WHERE [THERM] is not NULL AND [TEMP] is not NULL AND [DIGITS] is not NULL AND [VAL_STATUS]='Validated' AND [TM_STAMP] between '1/1/2022'and '1/1/2023'
rs1 = cur1.execute(qry1) #runs the query

In [ ]:

headers = ['ID_PIEZO','TM_STAMP','DIGITS','AMP','REPORTABLE']

# Replace rs1.fetchall() with the actual data
rs1_data = rs1.fetchall()

bad_query = pd.DataFrame([list(row) for row in rs1_data], columns=headers)


# Convert the column to numeric, coerce errors (i.e., non-numeric values) to NaN
#bad_query['DIGITS'] = pd.to_numeric(bad_query['DIGITS'], errors='coerce')

# Now replace NaN values with 0
bad_query['DIGITS'].fillna(-99, inplace=True)

bad_query['AMP'].fillna(-9, inplace=True)


In [ ]:

Flag_df=pd.read_csv('2yr.txt', delimiter=',')



# Find the ids in df1 that are not in df2
#ids_to_keep = set(bad_query['ID_PIEZO']) - set(Flag_df['ID_PIEZO'])

# Select the rows with these ids
#bad_query = bad_query[bad_query['ID_PIEZO'].isin(ids_to_keep)]


In [ ]:
Flag_df

In [ ]:
# importing os module
# Directory
import shutil
import os
directory = "TRAINING_FIGURES"

 

dir ="C:/Temp/TRAINING_FIGURES"
if os.path.exists(dir):
    shutil.rmtree(dir)
os.makedirs(dir)

In [ ]:
import matplotlib.pyplot as plt

unique_ids = Flag_df['ID_PIEZO'].unique()

for id in unique_ids:
    df_id = Flag_df[Flag_df['ID_PIEZO'] == id]
      # Create a new figure and two subplots, sharing x-axis
    fig, ax1 = plt.subplots()
 
    # First subplot for 'X' values
    ax1.plot(df_id['TM_STAMP'], df_id['DIGITS'], 'bo')
    ax1.set_xlabel('Time')
    plt.xticks(rotation=90)
    # Make the y-axis label and line color blue to match the line color.
    ax1.set_ylabel('DIGITS', color='b')
    ax1.tick_params('y', colors='b')
    if (max(df_id['DIGITS'])-min(df_id['DIGITS']))<10:
        ax1.set_ylim((min(df_id['DIGITS'])-25, max(df_id['DIGITS'])+25))

    # Second subplot for 'Y' values, sharing the same x-axis
    ax2 = ax1.twinx()
    ax2.plot(df_id['TM_STAMP'], df_id['AMP'], 'ro')
    # Make the y-axis label and line color red to match the line color.
    ax2.set_ylabel('AMP', color='r')
    ax2.tick_params('y', colors='r')
    ax2.set_ylim((-10, 10))
    
     # Create a third axis sharing the same x-axis
    ax3 = ax1.twinx()

    # Offset the third axis
    ax3.spines['right'].set_position(('outward', 60))
    
    # Plot data on the third axis
    ax3.plot(df_id['TM_STAMP'],df_id['REPORTABLE'],'yx', markersize=3, label='Actual_labels')
    ax3.set_ylabel('REPORTABLES', color='black')
    ax3.tick_params('y', colors='black')
    ax3.set_ylim((-0.1,1.1))
    # Add a title to the plot
    plt.title(f"Data for ID_PIEZO = {id}")

    fig.tight_layout()  # This ensures the right y-label is not cut off
    plt.show()
    fig.savefig('C:\Temp\TRAINING_FIGURES\{0}.png'.format(id), bbox_inches='tight',dpi=300)

In [ ]:
bad_query.to_csv('test_data.txt', sep=',', index=True)